# Budget-controller training and strategy evaluation

This notebook trains the lightweight context-budget policy and compares it with full-context, fixed-k, and elbow baselines. The bundled dataset is for pipeline verification; use the CLI's QASPER options for the final reported experiment.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from adacontext.config import load_config
from adacontext.datasets import load_sample_dataset
from adacontext.evaluation import evaluate_strategies
from adacontext.pipeline import AdaptiveContextPipeline
from adacontext.training import train_budget_controller

In [ ]:
config = load_config(overrides={'retrieval': {'embedding_backend': 'tfidf'}})
documents, questions = load_sample_dataset(ROOT)
train_questions = [q for q in questions if q.split == 'train']
test_questions = [q for q in questions if q.split == 'test']
training_pipeline = AdaptiveContextPipeline(config).index_documents(documents)
training_report = train_budget_controller(training_pipeline, train_questions, config['adaptive']['budget_model_path'])
training_report

In [ ]:
# Recreate the pipeline so it loads the newly trained controller.
pipeline = AdaptiveContextPipeline(config).index_documents(documents)
details, summary = evaluate_strategies(pipeline, test_questions, output_dir=ROOT / 'reports' / 'generated' / 'notebook_evaluation')
summary

In [ ]:
import plotly.express as px
px.scatter(summary, x='median_selected_tokens', y='token_f1', color='strategy', size='evidence_recall', hover_data=['mean_token_reduction', 'mean_total_ms'], title='Quality-cost Pareto frontier')